In [ ]:
from pathlib import Path

import hashlib
import json
import random
import time

import numpy as np
import pandas as pd

import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    CSVLogger,
)


SEED = 12345

IMAGE_SIZE = (128, 128)
BATCH_SIZE = 32
NUM_CLASSES = 35

FINETUNE_EPOCHS = 400
FINETUNE_PATIENCE = 200
FINETUNE_LEARNING_RATE = 1e-7

RUN_NAME = "mobilenet_clean_split_full_unfreeze"

OUTPUT_DIR = Path("/kaggle/working") / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

print("\nRun:", RUN_NAME)
print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Classes:", NUM_CLASSES)
print("Fine-tuning epochs:", FINETUNE_EPOCHS)
print("Fine-tuning patience:", FINETUNE_PATIENCE)
print("Fine-tuning learning rate:", FINETUNE_LEARNING_RATE)
print("Output directory:", OUTPUT_DIR)

In [ ]:
KAGGLE_INPUT = Path("/kaggle/input")


MANIFEST_PATH = next(
    KAGGLE_INPUT.rglob("zoolake_clean_split_manifest.csv")
)

CLASS_NAMES_PATH = next(
    KAGGLE_INPUT.rglob("zoolake_class_names.json")
)

DATASET_ROOT = next(
    KAGGLE_INPUT.rglob("zooplankton_0p5x")
)

FROZEN_MODEL_PATH = next(
    path
    for path in KAGGLE_INPUT.rglob("best_frozen_model.keras")
    if "mobilenet" in str(path).lower()
)

FROZEN_HISTORY_PATH = next(
    path
    for path in KAGGLE_INPUT.rglob("frozen_history.csv")
    if "mobilenet" in str(path).lower()
)


manifest_df = pd.read_csv(MANIFEST_PATH)

with open(
    CLASS_NAMES_PATH,
    "r",
    encoding="utf-8",
) as file:
    CLASS_NAMES = json.load(file)


manifest_df["filepath"] = manifest_df[
    "relative_filepath"
].apply(
    lambda path: str(DATASET_ROOT / path)
)


train_df = manifest_df[
    manifest_df["split"] == "train"
].copy()

validation_df = manifest_df[
    manifest_df["split"] == "validation"
].copy()

test_df = manifest_df[
    manifest_df["split"] == "test"
].copy()


print("Training:", len(train_df))
print("Validation:", len(validation_df))
print("Test:", len(test_df))
print("Classes:", len(CLASS_NAMES))

print("\nFrozen model:")
print(FROZEN_MODEL_PATH)

print("\nFrozen history:")
print(FROZEN_HISTORY_PATH)

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    rotation_range=180,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.20,
    shear_range=10,
)

eval_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
)


train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col="filepath",
    y_col="label",
    classes=CLASS_NAMES,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True,
    seed=SEED,
    interpolation="lanczos",
)

validation_generator = eval_datagen.flow_from_dataframe(
    dataframe=validation_df,
    x_col="filepath",
    y_col="label",
    classes=CLASS_NAMES,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False,
    interpolation="lanczos",
)

test_generator = eval_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col="filepath",
    y_col="label",
    classes=CLASS_NAMES,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False,
    interpolation="lanczos",
)

In [ ]:
model = tf.keras.models.load_model(
    FROZEN_MODEL_PATH
)


validation_generator.reset()

frozen_validation_results = model.evaluate(
    validation_generator,
    return_dict=True,
    verbose=1,
)


print("\nLoaded frozen model:")
print("Loss:", round(frozen_validation_results["loss"], 4))
print("Accuracy:", round(frozen_validation_results["accuracy"], 4))
print(
    "Top-2 accuracy:",
    round(frozen_validation_results["top2_accuracy"], 4),
)

In [ ]:
for layer in model.layers:
    layer.trainable = True


model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=FINETUNE_LEARNING_RATE,
    ),
    loss="categorical_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.TopKCategoricalAccuracy(
            k=2,
            name="top2_accuracy",
        ),
    ],
)


batch_normalization_layers = [
    layer
    for layer in model.layers
    if isinstance(
        layer,
        tf.keras.layers.BatchNormalization,
    )
]


print("Total layers:", len(model.layers))

print(
    "Trainable layers:",
    sum(layer.trainable for layer in model.layers),
)

print(
    "BatchNormalization layers:",
    len(batch_normalization_layers),
)

print(
    "Trainable BatchNormalization layers:",
    sum(
        layer.trainable
        for layer in batch_normalization_layers
    ),
)

print(
    "Learning rate:",
    float(model.optimizer.learning_rate.numpy()),
)

In [ ]:
BEST_WEIGHTS_PATH = (
    OUTPUT_DIR / "best_finetuned.weights.h5"
)

BEST_MODEL_PATH = (
    OUTPUT_DIR / "best_finetuned_model.keras"
)

FINETUNE_HISTORY_PATH = (
    OUTPUT_DIR / "finetuned_history.csv"
)

MAX_TRAINING_HOURS = 11


class TimeLimitCallback(tf.keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.start_time = time.time()

    def on_epoch_end(self, epoch, logs=None):
        elapsed_hours = (
            time.time() - self.start_time
        ) / 3600

        if elapsed_hours >= MAX_TRAINING_HOURS:
            print(
                "\nTraining stopped safely "
                "because the time limit was reached."
            )

            self.model.stop_training = True


callbacks = [
    ModelCheckpoint(
        filepath=str(BEST_WEIGHTS_PATH),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        save_weights_only=True,
        verbose=1,
    ),

    EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=FINETUNE_PATIENCE,
        restore_best_weights=False,
        verbose=1,
    ),

    CSVLogger(
        str(FINETUNE_HISTORY_PATH)
    ),
    TimeLimitCallback(),
]


print("Best weights:", BEST_WEIGHTS_PATH)
print("Best model:", BEST_MODEL_PATH)
print("Fine-tuning history:", FINETUNE_HISTORY_PATH)

print("\nCheckpoint monitor: val_loss")
print("Early-stopping patience:", FINETUNE_PATIENCE)

In [ ]:
finetune_start_time = time.time()


history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=FINETUNE_EPOCHS,
    callbacks=callbacks,
    verbose=2,
)


finetune_seconds = time.time() - finetune_start_time
epochs_completed = len(history.history["loss"])


training_time = {
    "epochs_completed": epochs_completed,
    "total_seconds": finetune_seconds,
    "total_hours": finetune_seconds / 3600,
    "average_seconds_per_epoch": (
        finetune_seconds / epochs_completed
    ),
}


with open(
    OUTPUT_DIR / "training_time.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        training_time,
        file,
        indent=2,
    )


print("\nFine-tuning completed")
print("Epochs completed:", epochs_completed)
print(
    "Total hours:",
    round(finetune_seconds / 3600, 2),
)
print(
    "Average seconds per epoch:",
    round(finetune_seconds / epochs_completed, 2),
)

In [ ]:

LAST_MODEL_PATH = (
    OUTPUT_DIR / "last_finetuned_model.keras"
)

model.save(LAST_MODEL_PATH)

print("Last training state saved:")
print(LAST_MODEL_PATH)


model.load_weights(BEST_WEIGHTS_PATH)

model.save(BEST_MODEL_PATH)


frozen_history_df = pd.read_csv(
    FROZEN_HISTORY_PATH
)

finetuned_history_df = pd.read_csv(
    FINETUNE_HISTORY_PATH
)


best_row_index = finetuned_history_df[
    "val_loss"
].idxmin()

best_finetune_epoch = (
    int(
        finetuned_history_df.loc[
            best_row_index,
            "epoch",
        ]
    )
    + 1
)

best_global_epoch = (
    len(frozen_history_df)
    + best_finetune_epoch
)

best_val_loss = float(
    finetuned_history_df.loc[
        best_row_index,
        "val_loss",
    ]
)


best_epoch_info = {
    "best_finetune_epoch": best_finetune_epoch,
    "best_global_epoch": best_global_epoch,
    "best_validation_loss": best_val_loss,
}


with open(
    OUTPUT_DIR / "best_epoch.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        best_epoch_info,
        file,
        indent=2,
    )


print("Best weights loaded:")
print(BEST_WEIGHTS_PATH)

print("\nComplete best model saved:")
print(BEST_MODEL_PATH)

print("\nBest fine-tuning epoch:", best_finetune_epoch)
print("Best global epoch:", best_global_epoch)
print("Best validation loss:", round(best_val_loss, 4))

In [ ]:
frozen_history_df["phase"] = "frozen"
frozen_history_df["global_epoch"] = np.arange(
    1,
    len(frozen_history_df) + 1,
)


finetuned_history_df["phase"] = "fine_tuning"
finetuned_history_df["global_epoch"] = (
    len(frozen_history_df)
    + np.arange(
        1,
        len(finetuned_history_df) + 1,
    )
)


combined_history_df = pd.concat(
    [
        frozen_history_df,
        finetuned_history_df,
    ],
    ignore_index=True,
)


COMBINED_HISTORY_PATH = (
    OUTPUT_DIR / "combined_history.csv"
)

combined_history_df.to_csv(
    COMBINED_HISTORY_PATH,
    index=False,
)


print("Combined history saved:")
print(COMBINED_HISTORY_PATH)

print("\nFrozen epochs:", len(frozen_history_df))
print(
    "Fine-tuning epochs:",
    len(finetuned_history_df),
)
print(
    "Total epochs:",
    len(combined_history_df),
)

In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
)


test_generator.reset()

test_results = model.evaluate(
    test_generator,
    return_dict=True,
    verbose=1,
)


test_generator.reset()

y_prob = model.predict(
    test_generator,
    verbose=1,
)

y_true = test_generator.classes
y_pred = np.argmax(y_prob, axis=1)


metrics = {
    "accuracy": float(
        test_results["accuracy"]
    ),
    "macro_precision": float(
        precision_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        )
    ),
    "macro_recall": float(
        recall_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        )
    ),
    "macro_f1": float(
        f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        )
    ),
    "top2_accuracy": float(
        test_results["top2_accuracy"]
    ),
    "test_loss": float(
        test_results["loss"]
    ),
    "best_finetune_epoch": (
        best_finetune_epoch
    ),
    "training_hours": float(
        training_time["total_hours"]
    ),
    "total_parameters": int(
        model.count_params()
    ),
}


METRICS_PATH = OUTPUT_DIR / "metrics.json"

with open(
    METRICS_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metrics,
        file,
        indent=2,
    )


PREDICTIONS_PATH = (
    OUTPUT_DIR / "predictions.npz"
)

np.savez_compressed(
    PREDICTIONS_PATH,
    y_true=y_true,
    y_pred=y_pred,
    y_prob=y_prob,
    class_names=np.array(CLASS_NAMES),
    image_names=test_df["image_name"].to_numpy(),
)


predictions_df = pd.DataFrame({
    "image_name": test_df[
        "image_name"
    ].to_numpy(),

    "true_label": [
        CLASS_NAMES[index]
        for index in y_true
    ],

    "predicted_label": [
        CLASS_NAMES[index]
        for index in y_pred
    ],

    "confidence": np.max(
        y_prob,
        axis=1,
    ),
})


PREDICTIONS_CSV_PATH = (
    OUTPUT_DIR / "test_predictions.csv"
)

predictions_df.to_csv(
    PREDICTIONS_CSV_PATH,
    index=False,
)


print("\nTest results:")

for name, value in metrics.items():
    print(name, ":", value)

print("\nSaved:", METRICS_PATH)
print("Saved:", PREDICTIONS_PATH)
print("Saved:", PREDICTIONS_CSV_PATH)

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
)


report = classification_report(
    y_true,
    y_pred,
    labels=np.arange(NUM_CLASSES),
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0,
)


report_df = pd.DataFrame(
    report
).transpose()

report_df.index.name = "class"


REPORT_PATH = (
    OUTPUT_DIR / "classification_report.csv"
)

report_df.to_csv(
    REPORT_PATH
)


confusion_counts = confusion_matrix(
    y_true,
    y_pred,
    labels=np.arange(NUM_CLASSES),
)


confusion_normalized = (
    confusion_counts.astype(float)
    / confusion_counts.sum(
        axis=1,
        keepdims=True,
    )
)

confusion_normalized = np.nan_to_num(
    confusion_normalized
)


CONFUSION_COUNTS_PATH = (
    OUTPUT_DIR / "confusion_matrix_counts.csv"
)

CONFUSION_NORMALIZED_PATH = (
    OUTPUT_DIR / "confusion_matrix_normalized.csv"
)


pd.DataFrame(
    confusion_counts,
    index=CLASS_NAMES,
    columns=CLASS_NAMES,
).to_csv(
    CONFUSION_COUNTS_PATH
)

pd.DataFrame(
    confusion_normalized,
    index=CLASS_NAMES,
    columns=CLASS_NAMES,
).to_csv(
    CONFUSION_NORMALIZED_PATH
)


print("Saved:", REPORT_PATH)
print("Saved:", CONFUSION_COUNTS_PATH)
print("Saved:", CONFUSION_NORMALIZED_PATH)

In [ ]:
import shutil


ZIP_PATH = shutil.make_archive(
    str(
        Path("/kaggle/working")
        / f"{RUN_NAME}_results"
    ),
    "zip",
    root_dir=OUTPUT_DIR,
)


print("Results folder:")
print(OUTPUT_DIR)

print("\nResults ZIP:")
print(ZIP_PATH)